In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")


Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
from text_processing import *
from utils import *
hotpot_file_candidates = [
    REPO_ROOT / "jupyter_notebooks" / "hotpot_dev_distractor_v1.json",
    REPO_ROOT / "hotpot_dev_distractor_v1.json",
]
file_path = next((str(path) for path in hotpot_file_candidates if path.exists()), str(hotpot_file_candidates[0]))
print(f"Using HotpotQA file: {file_path}")
documents, samples = build_hotpot_retrieval_dataset(file_path, num_samples=500)
# print("Example document:\n")
# print("Title:", documents[0]["title"])
# print("Text:", documents[0]["text"][:200])

Using HotpotQA file: /home/xiaoyue/LiteSemRAG/jupyter_notebooks/hotpot_dev_distractor_v1.json
Loading cached dataset...
Loaded 4937 documents
Loaded 500 samples


In [3]:
for index in range(40):
    print("Question:",samples[index]['question'])
    for idx in samples[index]['gold_doc_ids']:
        print("Document:", documents[idx]['text'])

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Document: Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director.
Document: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Document: Kiss and Tell is a 1945 American comedy film starring then 17-year-old Shirley Temple as Corliss Archer.  In the film, two teenage girls cause their respective parents much concern when they start to become interested in boys.  The parents' bickering about which girl is the worse influence causes more problems than it solves.
Docum

In [2]:
import RAG_graph
graph_database = RAG_graph.ProtoGraphRAG.load_data_split("rag_multihop_database.pkl")

Loading text encoder models in device: GPU


In [4]:
import RAG_graph
graph_database = RAG_graph.ProtoGraphRAG(
    text_embed_dim=1024,
    df_ratio=0.9,
    buffer_size=100,
    chunk_size=256,
    remove_duplicate_token=True,
    device="cuda",
    plot_embeds=False,
    exhaustive_proto_description_evaluation=True
)
graph_database.index_json(documents,batch_size=4)
graph_database.finalize()
graph_database.print_memory_size()
#graph_database.save_data_split("rag_multihop_database_explain.pkl")

Loading text encoder models in device: GPU


/home/xiaoyue/anaconda3/envs/llm_graph/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

CPU preprocessed: 4937/4937 | GPU encoded: 4937/4937 | CPU processed: 4937/4937
[891.5010s] Index 4937 documents. Index time: 891.5010s
[896.8945s] Finalize started.
[896.8966s] Computed average chunk length.
finalize_token_nodes: 64424/64424
[910.8856s] Finished token node finalization.
merge_duplicate_description_proto_nodes: 40/40
[1214.0514s] Finished merging proto nodes by description.
[1214.1913s] Finished assigning token and proto IDF.
[1214.4625s] Finished computing proto BM25.
[1214.7839s] Finished building query database.
[1215.5034s] Finished building phrase query index.
[1215.6797s] Finished building chunk-to-proto edges.
[1215.6845s] Finalizing completed.
memory size: 136.50 MB


In [5]:
graph_database.show_proto_description_logs()

In [5]:
graph_database.show_multi_proto_token_nodes(min_proto_count=2,
            max_sentences_per_proto=30,
            as_html=True,
            token_contains=None,
            sort_by="proto_count",
            max_token_nodes=50,
            max_protos_per_token=10,
            max_examples_per_token=50,
            open_details=False)

In [ ]:
import time
correct = 0
mrr_sum = 0
start = time.time()
for sample_idx, sample in enumerate(samples[:500]):
    num_correct = 0
    #print(sample['question'])
    _, retrieved_chunk_node_id, cog = graph_database.multi_level_query(sample['question'], top_k_chunk=10, top_k_each_isolated_chunk=2, isolate_retrieve_mode='sequential', isolate_chunk_ratio=0.5,print_important_tokens=False)
    #rerank_chunks, retrieved_chunk_node_id = graph_database.broad_search_query(sample['question'],top_k=10)
    all_titles = []
    for idx in retrieved_chunk_node_id:
        all_titles.append(graph_database.chunk_nodes[idx].doc_node.doc_name)
    correct_titles = [documents[index]['title'] for index in sample['gold_doc_ids']]
    mrr = mrr_for_one_query_titles(all_titles, correct_titles, k=10)
    mrr_sum += mrr
    for answer in correct_titles:
        if answer in all_titles:
            correct = correct + 1
            num_correct += 1
    #print(f"Correct: {num_correct}")
    #print("=========================================================================")
    if num_correct < 2:
        print(f"question {sample_idx}:{sample['question']}, correct:{num_correct}, mrr:{mrr}")
    #print(f"question {sample_idx}:{sample['question']}, correct:{num_correct}")
print(correct/1000)
print(mrr_sum/500)
end = time.time()
print(f"运行时间：{end - start:.6f} 秒")

In [ ]:
inspect_index = 57
print(samples[inspect_index]['question'])
question = samples[inspect_index]['question']
retrieved_chunk, retrieved_chunk_node_id, cog = graph_database.multi_level_query(clean_text(question), top_k_chunk=10, isolate_retrieve_mode='sequential', isolate_chunk_ratio=0.2,print_important_tokens=True)
for index, chunk in enumerate(retrieved_chunk):
    print(f"Retrieved {index} :{chunk}")
print("====================================================================")
for index in samples[inspect_index]['gold_doc_ids']:
    print(documents[index]['text'])